# 121: QFO 2020 — Kmerseek All-vs-All Analysis

Analyze the kmerseek all-vs-all search results from the Quest for Orthologs (QfO) 2020 benchmark.

**Pipeline:** `nextflow-runs/qfo/2020/main.nf`  
**Parameters:** k=24, moltype=hp, scaled=1  
**Proteomes:** 78 species (7 Archaea, 23 Bacteria, 48 Eukaryota)  
**Pairs run:** 3001 pairwise comparisons  

## Goals
1. **Diagnose** why `kmerseek_k24_hp_qfo2020.orthoxml` is empty (zero ortholog groups)
2. **Explore** the search result columns and score distributions
3. **Identify** significant hits using containment / ANI thresholds
4. **Characterize** which species pairs show the most similarity
5. **Re-form** ortholog groups using containment-based criteria

In [1]:
import gzip
import os
import re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

sns.set_style("whitegrid")

WORK_DIR = Path("/Users/olga/code/2024-kmerseek-analysis/nextflow-runs/qfo/2020/work")
RESULTS_DIR = Path(
    "/Users/olga/code/2024-kmerseek-analysis/nextflow-runs/qfo/2020/results"
)
OUT_DIR = RESULTS_DIR  # save analysis outputs alongside pipeline outputs

PREFIX = "121_"
KSIZE = 24
MOLTYPE = "hp"

print(f"Work dir exists: {WORK_DIR.exists()}")
print(f"Results dir exists: {RESULTS_DIR.exists()}")

Work dir exists: True
Results dir exists: True


## 1. Diagnose the Empty OrthoXML

The pipeline produced `kmerseek_k24_hp_qfo2020.orthoxml` with zero ortholog groups.
`format_orthoxml.py` looks for columns `poisson_pvalue`, `prob_overlap`, or `prob_overlap_adjusted`.
Let's check what columns are actually in the search results.

In [2]:
# Read the orthoxml to confirm it's empty
with open(RESULTS_DIR / f"kmerseek_k{KSIZE}_{MOLTYPE}_qfo2020.orthoxml") as fh:
    content = fh.read()
print("=== OrthoXML contents ===")
print(content)
print()

# Check columns in a sample search result CSV
sample_csvs = list(WORK_DIR.glob("**/*.csv.gz"))
print(f"Total CSV.gz files in work dir: {len(sample_csvs):,}")

sample_csv = sample_csvs[0]
print(f"\nSample file: {sample_csv.name}")
with gzip.open(sample_csv, "rt") as fh:
    header = fh.readline().strip()
columns = header.split(",")
print(f"Columns ({len(columns)}):")
for col in columns:
    print(f"  {col}")

# Check for the p-value columns that format_orthoxml.py looks for
PVAL_COLUMNS = ("poisson_pvalue", "prob_overlap", "prob_overlap_adjusted")
found = [c for c in PVAL_COLUMNS if c in columns]
print(f"\nExpected p-value columns: {PVAL_COLUMNS}")
print(f"Found in CSV: {found}")
if not found:
    print("\n⚠️  None of the expected p-value columns are present.")
    print("   format_orthoxml.py skips files without them → empty OrthoXML.")

=== OrthoXML contents ===
<?xml version="1.0" encoding="utf-8"?>
<orthoXML xmlns="http://orthoXML.org/2011/" version="0.3" origin="kmerseek" originVersion="k24_hp_scaled1_p0.05">
  <groups />
</orthoXML>

Total CSV.gz files in work dir: 9,007

Sample file: UP000000589_10090_vs_UP000001570_224308.csv.gz
Columns (22):
  query_name
  query_md5
  match_name
  containment
  intersect_hashes
  ksize
  scaled
  moltype
  match_md5
  jaccard
  max_containment
  average_abund
  median_abund
  std_abund
  query_containment_ani
  match_containment_ani
  average_containment_ani
  max_containment_ani
  n_weighted_found
  total_weighted_hashes
  containment_target_in_query
  f_weighted_target_in_query

Expected p-value columns: ('poisson_pvalue', 'prob_overlap', 'prob_overlap_adjusted')
Found in CSV: []

⚠️  None of the expected p-value columns are present.
   format_orthoxml.py skips files without them → empty OrthoXML.


In [3]:
# Read a sample file to inspect score distributions
df_sample = pl.read_csv(sample_csv, n_rows=50_000)

score_cols = [
    "containment",
    "jaccard",
    "max_containment",
    "intersect_hashes",
    "query_containment_ani",
    "match_containment_ani",
    "average_containment_ani",
]
existing_score_cols = [c for c in score_cols if c in df_sample.columns]

print("Score column summary (first 50k rows):")
print(df_sample.select(existing_score_cols).describe())

print(f"\nSample file: {sample_csv.name}")
# Show a few rows
df_sample.select(["query_name", "match_name"] + existing_score_cols[:4]).head(3)

Score column summary (first 50k rows):
shape: (9, 8)
┌────────────┬────────────┬──────────┬────────────┬────────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ containmen ┆ jaccard  ┆ max_contai ┆ intersect_ ┆ query_con ┆ match_con ┆ average_c │
│ ---        ┆ t          ┆ ---      ┆ nment      ┆ hashes     ┆ tainment_ ┆ tainment_ ┆ ontainmen │
│ str        ┆ ---        ┆ f64      ┆ ---        ┆ ---        ┆ ani       ┆ ani       ┆ t_ani     │
│            ┆ f64        ┆          ┆ f64        ┆ f64        ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆          ┆            ┆            ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪══════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 50000.0    ┆ 50000.0  ┆ 50000.0    ┆ 50000.0    ┆ 50000.0   ┆ 50000.0   ┆ 50000.0   │
│ null_count ┆ 0.0        ┆ 0.0      ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ mean       ┆ 0.005193   ┆ 0.002555 ┆

query_name,match_name,containment,jaccard,max_containment,intersect_hashes
str,str,f64,f64,f64,i64
"""sp|A2AL36|CNTRL_MOUSE Centriol…","""sp|O31972|YOZP_BACSU Uncharact…",0.000433,0.000418,0.011628,1
"""sp|A2AMT1|BFSP1_MOUSE Filensin…","""sp|O31972|YOZP_BACSU Uncharact…",0.003096,0.00274,0.023256,2
"""sp|A2ASS6|TITIN_MOUSE Titin OS…","""sp|O31972|YOZP_BACSU Uncharact…",0.000119,0.000118,0.046512,4


## 2. Collect Summary Statistics Across All Pairs

Reading all 3001 CSV files in full would be very large. Instead, we collect:
- The **best hit** per query (max containment)
- Per-file statistics: n_rows, max/median containment, max ANI

This gives us a compact view of all species pairs.

In [4]:
PAIRWISE_STATS_PATH = OUT_DIR / f"{PREFIX}pairwise_stats.parquet"

# QfO 2020 species lookup (taxid -> species_name)
QFO_SPECIES = {
    36329: "Plasmodium falciparum",
    7070: "Tribolium castaneum",
    188937: "Methanosarcina acetivorans",
    83332: "Mycobacterium tuberculosis",
    10090: "Mus musculus",
    7719: "Ciona intestinalis",
    81824: "Monosiga brevicollis",
    321614: "Phaeosphaeria nodorum",
    9595: "Gorilla gorilla gorilla",
    374847: "Korarchaeum cryptofilum",
    44689: "Dictyostelium discoideum",
    7165: "Anopheles gambiae",
    9606: "Homo sapiens",
    284812: "Schizosaccharomyces pombe",
    243232: "Methanocaldococcus jannaschii",
    9598: "Pan troglodytes",
    208964: "Pseudomonas aeruginosa",
    9615: "Canis lupus familiaris",
    684364: "Batrachochytrium dendrobatidis",
    8090: "Oryzias latipes",
    243274: "Thermotoga maritima",
    8364: "Xenopus tropicalis",
    184922: "Giardia intestinalis",
    418459: "Puccinia graminis",
    272561: "Chlamydia trachomatis",
    100226: "Streptomyces coelicolor",
    189518: "Leptospira interrogans",
    83333: "Escherichia coli",
    559292: "Saccharomyces cerevisiae",
    122586: "Neisseria meningitidis",
    243230: "Deinococcus radiodurans",
    3218: "Physcomitrella patens",
    665079: "Sclerotinia sclerotiorum",
    224911: "Bradyrhizobium diazoefficiens",
    164328: "Phytophthora ramorum",
    1111708: "Synechocystis sp. PCC 6803",
    7227: "Drosophila melanogaster",
    324602: "Chloroflexus aurantiacus",
    7955: "Danio rerio",
    35128: "Thalassiosira pseudonana",
    224308: "Bacillus subtilis",
    6945: "Ixodes scapularis",
    251221: "Gloeobacter violaceus",
    224324: "Aquifex aeolicus",
    214684: "Cryptococcus neoformans",
    436308: "Nitrosopumilus maritimus",
    64091: "Halobacterium salinarum",
    284591: "Yarrowia lipolytica",
    85962: "Helicobacter pylori",
    5722: "Trichomonas vaginalis",
    9913: "Bos taurus",
    367110: "Neurospora crassa",
    6239: "Caenorhabditis elegans",
    190304: "Fusobacterium nucleatum",
    4577: "Zea mays",
    273057: "Saccharolobus solfataricus",
    3702: "Arabidopsis thaliana",
    243090: "Rhodopirellula baltica",
    13616: "Monodelphis domestica",
    5664: "Leishmania major",
    45351: "Nematostella vectensis",
    9031: "Gallus gallus",
    237561: "Candida albicans",
    39947: "Oryza sativa subsp. japonica",
    330879: "Neosartorya fumigata",
    243231: "Geobacter sulfurreducens",
    69014: "Thermococcus kodakarensis",
    243273: "Mycoplasma genitalium",
    3055: "Chlamydomonas reinhardtii",
    6412: "Helobdella robusta",
    5888: "Paramecium tetraurelia",
    10116: "Rattus norvegicus",
    515635: "Dictyoglomus turgidum",
    237631: "Ustilago maydis",
    7739: "Branchiostoma floridae",
    226186: "Bacteroides thetaiotaomicron",
    289376: "Thermodesulfovibrio yellowstonii",
    7918: "Lepisosteus oculatus",
}


def parse_filename(csv_path):
    """Return (query_proteome, query_taxid, target_proteome, target_taxid) from filename."""
    stem = csv_path.stem.replace(".csv", "")  # handle .csv.gz
    left, right = stem.split("_vs_")
    qproteome, qtaxid = left.rsplit("_", 1)
    tproteome, ttaxid = right.rsplit("_", 1)
    return qproteome, int(qtaxid), tproteome, int(ttaxid)


if PAIRWISE_STATS_PATH.exists():
    pair_stats = pl.read_parquet(PAIRWISE_STATS_PATH)
    print(f"Loaded cached stats: {pair_stats.shape}")
else:
    rows = []
    csv_files = list(WORK_DIR.glob("**/*.csv.gz"))
    print(f"Processing {len(csv_files):,} CSV files...")

    for i, csv_path in enumerate(csv_files):
        try:
            qproteome, qtaxid, tproteome, ttaxid = parse_filename(csv_path)
        except Exception:
            continue

        try:
            df = pl.read_csv(
                csv_path,
                columns=[
                    "containment",
                    "jaccard",
                    "max_containment",
                    "intersect_hashes",
                    "average_containment_ani",
                    "max_containment_ani",
                ],
            )
        except Exception as e:
            continue

        if len(df) == 0:
            continue

        rows.append(
            {
                "query_proteome": qproteome,
                "query_taxid": qtaxid,
                "target_proteome": tproteome,
                "target_taxid": ttaxid,
                "query_species": QFO_SPECIES.get(qtaxid, f"taxid_{qtaxid}"),
                "target_species": QFO_SPECIES.get(ttaxid, f"taxid_{ttaxid}"),
                "n_hits": len(df),
                "max_containment_max": float(df["max_containment"].max() or 0),
                "max_containment_median": float(df["max_containment"].median() or 0),
                "containment_max": float(df["containment"].max() or 0),
                "containment_median": float(df["containment"].median() or 0),
                "jaccard_max": float(df["jaccard"].max() or 0),
                "jaccard_median": float(df["jaccard"].median() or 0),
                "intersect_hashes_max": int(df["intersect_hashes"].max() or 0),
                "intersect_hashes_median": float(df["intersect_hashes"].median() or 0),
                "ani_max": float(df["average_containment_ani"].max() or 0),
                "ani_median": float(df["average_containment_ani"].median() or 0),
                # Fraction of hits exceeding containment thresholds
                "n_containment_ge_01": int((df["max_containment"] >= 0.1).sum()),
                "n_containment_ge_05": int((df["max_containment"] >= 0.5).sum()),
                "n_ani_ge_70": int((df["average_containment_ani"] >= 0.70).sum()),
                "n_ani_ge_80": int((df["average_containment_ani"] >= 0.80).sum()),
            }
        )

        if (i + 1) % 500 == 0:
            print(f"  {i + 1}/{len(csv_files)}")

    pair_stats = pl.DataFrame(rows)
    pair_stats.write_parquet(PAIRWISE_STATS_PATH)
    print(f"Saved stats for {len(pair_stats):,} pairs to {PAIRWISE_STATS_PATH}")

print(f"Pair stats shape: {pair_stats.shape}")
pair_stats.head(5)

Processing 9,007 CSV files...
  500/9007


KeyboardInterrupt: 

## 3. Overall Hit Statistics

In [ ]:
total_hits = pair_stats["n_hits"].sum()
total_pairs = len(pair_stats)

n_any_01 = (pair_stats["n_containment_ge_01"] > 0).sum()
n_any_05 = (pair_stats["n_containment_ge_05"] > 0).sum()
n_any_ani70 = (pair_stats["n_ani_ge_70"] > 0).sum()
n_any_ani80 = (pair_stats["n_ani_ge_80"] > 0).sum()

total_sig_01 = pair_stats["n_containment_ge_01"].sum()
total_sig_05 = pair_stats["n_containment_ge_05"].sum()
total_ani70 = pair_stats["n_ani_ge_70"].sum()
total_ani80 = pair_stats["n_ani_ge_80"].sum()

print(f"Total species pairs:                   {total_pairs:,}")
print(f"Total pairwise hits (all):             {total_hits:,}")
print()
print(
    f"Hits with max_containment >= 0.10:     {total_sig_01:,} ({total_sig_01/total_hits*100:.2f}%)"
)
print(f"  Pairs with any such hit:             {n_any_01:,} / {total_pairs:,}")
print(f"Hits with max_containment >= 0.50:     {total_sig_05:,}")
print(f"  Pairs with any such hit:             {n_any_05:,} / {total_pairs:,}")
print()
print(f"Hits with avg ANI >= 70%:              {total_ani70:,}")
print(f"  Pairs with any such hit:             {n_any_ani70:,} / {total_pairs:,}")
print(f"Hits with avg ANI >= 80%:              {total_ani80:,}")
print(f"  Pairs with any such hit:             {n_any_ani80:,} / {total_pairs:,}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    "Distribution of per-pair maximum scores across all 3001 species pairs", fontsize=12
)

ax = axes[0]
ax.hist(
    pair_stats["max_containment_max"].to_numpy(),
    bins=50,
    color="steelblue",
    edgecolor="white",
)
ax.set_xlabel("Max containment (per pair)")
ax.set_ylabel("Number of species pairs")
ax.set_title("Max containment")
ax.axvline(0.1, color="red", linestyle="--", label="0.10 threshold")
ax.legend(fontsize=8)

ax = axes[1]
ax.hist(
    pair_stats["jaccard_max"].to_numpy(), bins=50, color="darkorange", edgecolor="white"
)
ax.set_xlabel("Max Jaccard (per pair)")
ax.set_title("Max Jaccard")

ax = axes[2]
ax.hist(pair_stats["ani_max"].to_numpy(), bins=50, color="seagreen", edgecolor="white")
ax.set_xlabel("Max average ANI (per pair)")
ax.set_title("Max average ANI")
ax.axvline(0.70, color="red", linestyle="--", label="70% ANI")
ax.axvline(0.80, color="purple", linestyle="--", label="80% ANI")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PREFIX}score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Top Species Pairs by Similarity

In [ ]:
# Top 20 pairs by max containment
top_pairs = (
    pair_stats.sort("max_containment_max", descending=True)
    .select(
        [
            "query_species",
            "target_species",
            "max_containment_max",
            "ani_max",
            "n_containment_ge_01",
            "n_containment_ge_05",
            "n_ani_ge_80",
            "n_hits",
        ]
    )
    .head(25)
)

print("Top 25 species pairs by max containment:")
top_pairs

In [ ]:
top20 = top_pairs.head(20).to_pandas()
top20["pair_label"] = (
    top20["query_species"].str.split().str[0]
    + " vs "
    + top20["target_species"].str.split().str[0]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

ax = axes[0]
colors = sns.color_palette("Blues_d", len(top20))
ax.barh(range(len(top20)), top20["max_containment_max"].values[::-1], color=colors)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20["pair_label"].values[::-1], fontsize=8)
ax.set_xlabel("Max containment")
ax.set_title("Top 20 pairs: max containment")
ax.axvline(0.1, color="red", linestyle="--", alpha=0.5)

ax = axes[1]
top20_ani = pair_stats.sort("ani_max", descending=True).head(20).to_pandas()
top20_ani["pair_label"] = (
    top20_ani["query_species"].str.split().str[0]
    + " vs "
    + top20_ani["target_species"].str.split().str[0]
)
colors_ani = sns.color_palette("Greens_d", len(top20_ani))
ax.barh(range(len(top20_ani)), top20_ani["ani_max"].values[::-1], color=colors_ani)
ax.set_yticks(range(len(top20_ani)))
ax.set_yticklabels(top20_ani["pair_label"].values[::-1], fontsize=8)
ax.set_xlabel("Max average ANI")
ax.set_title("Top 20 pairs: max ANI")
ax.axvline(0.70, color="red", linestyle="--", alpha=0.5, label="70% ANI")
ax.axvline(0.80, color="purple", linestyle="--", alpha=0.5, label="80% ANI")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PREFIX}top_species_pairs.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Kingdom-Level Comparison

Are within-kingdom pairs more similar than cross-kingdom pairs?

In [ ]:
# Assign kingdom from proteome directory (Archaea/Bacteria/Eukaryota based on filename prefix)
# We can infer this from the QFO species table — we'll do it by taxid ranges
# Or load from the data dir

DATA_DIR = Path(
    "/Users/olga/data/quest-for-orthologs/QfO_release_2020_04_with_updated_UP000008143"
)


def get_kingdom(proteome_id):
    """Determine kingdom from QFO directory structure."""
    for kingdom in ["Archaea", "Bacteria", "Eukaryota"]:
        kingdom_dir = DATA_DIR / kingdom
        if kingdom_dir.exists() and (kingdom_dir / f"{proteome_id}.fasta").exists():
            return kingdom
    return "Unknown"


# Build kingdom lookup once
kingdom_lookup = {}
if DATA_DIR.exists():
    for kingdom in ["Archaea", "Bacteria", "Eukaryota"]:
        kingdom_dir = DATA_DIR / kingdom
        if kingdom_dir.exists():
            for fasta in kingdom_dir.glob("UP*.fasta"):
                # Only canonical (not _additional, not _DNA)
                if "_additional" not in fasta.name and "_DNA" not in fasta.name:
                    proteome_id = fasta.stem.split("_")[0]
                    kingdom_lookup[proteome_id] = kingdom

print(f"Kingdom lookup: {len(kingdom_lookup)} proteomes")
from collections import Counter

print(Counter(kingdom_lookup.values()))

# Add kingdom columns to pair_stats
query_kingdoms = [
    kingdom_lookup.get(p, "Unknown") for p in pair_stats["query_proteome"].to_list()
]
target_kingdoms = [
    kingdom_lookup.get(p, "Unknown") for p in pair_stats["target_proteome"].to_list()
]

pair_stats = pair_stats.with_columns(
    [
        pl.Series("query_kingdom", query_kingdoms),
        pl.Series("target_kingdom", target_kingdoms),
    ]
)


# Create kingdom-pair label
def make_kingdom_pair(qk, tk):
    kingdoms = sorted([qk, tk])
    return f"{kingdoms[0]} vs {kingdoms[1]}"


kingdom_pairs = [
    make_kingdom_pair(qk, tk) for qk, tk in zip(query_kingdoms, target_kingdoms)
]
pair_stats = pair_stats.with_columns(pl.Series("kingdom_pair", kingdom_pairs))

pair_stats.group_by("kingdom_pair").agg(
    pl.len().alias("n_pairs"),
    pl.col("max_containment_max").mean().alias("mean_max_containment"),
    pl.col("max_containment_max").median().alias("median_max_containment"),
    pl.col("n_containment_ge_01").sum().alias("total_hits_ge_01"),
    pl.col("n_ani_ge_70").sum().alias("total_hits_ani70"),
).sort("median_max_containment", descending=True)

In [ ]:
import pandas as pd

pair_pd = pair_stats.to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

kingdom_pair_order = (
    pair_pd.groupby("kingdom_pair")["max_containment_max"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

ax = axes[0]
sns.boxplot(
    data=pair_pd,
    x="kingdom_pair",
    y="max_containment_max",
    order=kingdom_pair_order,
    ax=ax,
    showfliers=False,
    palette="Set2",
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8)
ax.set_xlabel("")
ax.set_ylabel("Max containment (per pair)")
ax.set_title("Max containment by kingdom pair")

ax = axes[1]
sns.boxplot(
    data=pair_pd,
    x="kingdom_pair",
    y="ani_max",
    order=kingdom_pair_order,
    ax=ax,
    showfliers=False,
    palette="Set2",
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8)
ax.set_xlabel("")
ax.set_ylabel("Max ANI (per pair)")
ax.set_title("Max ANI by kingdom pair")
ax.axhline(0.70, color="red", linestyle="--", alpha=0.5, label="70% ANI")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / f"{PREFIX}kingdom_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Deep Dive: High-Similarity Pairs

Load all hits with `max_containment >= 0.10` across all pairs to examine the strongly-similar proteins.

In [ ]:
STRONG_HITS_PATH = OUT_DIR / f"{PREFIX}strong_hits_containment_ge_01.parquet"

CONTAINMENT_THRESHOLD = 0.10

if STRONG_HITS_PATH.exists():
    strong_hits = pl.read_parquet(STRONG_HITS_PATH)
    print(f"Loaded cached strong hits: {strong_hits.shape}")
else:
    # Only load from pairs that have any hits above threshold
    pairs_with_hits = pair_stats.filter(pl.col("n_containment_ge_01") > 0)
    print(
        f"Pairs with max_containment >= {CONTAINMENT_THRESHOLD}: {len(pairs_with_hits):,}"
    )

    dfs = []
    for row in pairs_with_hits.iter_rows(named=True):
        # Find the CSV for this pair
        pattern = f"{row['query_proteome']}_{row['query_taxid']}_vs_{row['target_proteome']}_{row['target_taxid']}.csv.gz"
        csv_files_for_pair = list(WORK_DIR.glob(f"**/{pattern}"))
        if not csv_files_for_pair:
            continue

        try:
            df = (
                pl.read_csv(csv_files_for_pair[0])
                .filter(pl.col("max_containment") >= CONTAINMENT_THRESHOLD)
                .with_columns(
                    [
                        pl.lit(row["query_proteome"]).alias("query_proteome"),
                        pl.lit(row["query_taxid"]).alias("query_taxid"),
                        pl.lit(row["target_proteome"]).alias("target_proteome"),
                        pl.lit(row["target_taxid"]).alias("target_taxid"),
                        pl.lit(row["query_species"]).alias("query_species"),
                        pl.lit(row["target_species"]).alias("target_species"),
                        pl.lit(row["query_kingdom"]).alias("query_kingdom"),
                        pl.lit(row["target_kingdom"]).alias("target_kingdom"),
                    ]
                )
            )
            dfs.append(df)
        except Exception as e:
            print(f"  Error loading {pattern}: {e}")

    if dfs:
        strong_hits = pl.concat(dfs, how="diagonal")
        strong_hits.write_parquet(STRONG_HITS_PATH)
        print(f"Saved {len(strong_hits):,} strong hits to {STRONG_HITS_PATH}")
    else:
        strong_hits = pl.DataFrame()
        print("No strong hits found.")

print(f"Strong hits shape: {strong_hits.shape}")
if len(strong_hits):
    strong_hits.select(
        [
            "query_name",
            "match_name",
            "containment",
            "max_containment",
            "average_containment_ani",
            "query_species",
            "target_species",
        ]
    ).head(5)

In [ ]:
if len(strong_hits) > 0:
    print(
        f"Strong hits (max_containment >= {CONTAINMENT_THRESHOLD}): {len(strong_hits):,}"
    )
    print()

    # Count by kingdom pair
    kingdom_pair_col = [
        make_kingdom_pair(qk, tk)
        for qk, tk in zip(
            strong_hits["query_kingdom"].to_list(),
            strong_hits["target_kingdom"].to_list(),
        )
    ]
    strong_hits = strong_hits.with_columns(pl.Series("kingdom_pair", kingdom_pair_col))

    print("Strong hits by kingdom pair:")
    display(
        strong_hits.group_by("kingdom_pair")
        .agg(
            pl.len().alias("n_hits"),
            pl.col("max_containment").median().alias("median_max_containment"),
            pl.col("average_containment_ani").median().alias("median_ani"),
        )
        .sort("n_hits", descending=True)
    )

    print("\nTop species pairs by hit count:")
    display(
        strong_hits.group_by(["query_species", "target_species"])
        .agg(
            pl.len().alias("n_strong_hits"),
            pl.col("max_containment").max().alias("max_containment"),
            pl.col("average_containment_ani").max().alias("max_ani"),
        )
        .sort("n_strong_hits", descending=True)
        .head(20)
    )

## 7. Containment vs. ANI Score Relationship

In [ ]:
if len(strong_hits) > 0:
    # Sample to keep plot fast
    sample_size = min(len(strong_hits), 50_000)
    sh_sample = strong_hits.sample(sample_size, seed=42).to_pandas()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f"Strong hits (max_containment >= {CONTAINMENT_THRESHOLD}), n={len(strong_hits):,}",
        fontsize=11,
    )

    ax = axes[0]
    scatter = ax.scatter(
        sh_sample["max_containment"],
        sh_sample["average_containment_ani"],
        c=sh_sample["intersect_hashes"],
        norm=plt.matplotlib.colors.LogNorm(),
        s=2,
        alpha=0.3,
        cmap="viridis",
    )
    plt.colorbar(scatter, ax=ax, label="intersect_hashes (log)")
    ax.set_xlabel("max_containment")
    ax.set_ylabel("average_containment_ani")
    ax.set_title("Containment vs. ANI")

    ax = axes[1]
    ax.scatter(
        sh_sample["jaccard"],
        sh_sample["max_containment"],
        s=2,
        alpha=0.3,
        color="steelblue",
    )
    ax.set_xlabel("Jaccard")
    ax.set_ylabel("max_containment")
    ax.set_title("Jaccard vs. max_containment")

    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"{PREFIX}containment_ani_scatter.png", dpi=150, bbox_inches="tight"
    )
    plt.show()

## 8. Re-Form Ortholog Groups Using Containment Threshold

Since `format_orthoxml.py` produced an empty output (missing `poisson_pvalue` column),
we use `max_containment >= 0.10` as an alternative significance criterion and
reconstruct connected-component ortholog groups via union-find.

In [ ]:
if len(strong_hits) == 0:
    print("No strong hits — skipping group formation.")
else:

    class UnionFind:
        def __init__(self):
            self.parent = {}
            self.rank = {}

        def find(self, x):
            if x not in self.parent:
                self.parent[x] = x
                self.rank[x] = 0
            if self.parent[x] != x:
                self.parent[x] = self.find(self.parent[x])
            return self.parent[x]

        def union(self, x, y):
            rx, ry = self.find(x), self.find(y)
            if rx == ry:
                return
            if self.rank[rx] < self.rank[ry]:
                rx, ry = ry, rx
            self.parent[ry] = rx
            if self.rank[rx] == self.rank[ry]:
                self.rank[rx] += 1

    def parse_accession(seq_name):
        name = str(seq_name).strip().split()[0]
        parts = name.split("|")
        return parts[1] if len(parts) >= 2 else name

    uf = UnionFind()
    gene_species = {}  # accession -> (taxid, proteome)

    for row in strong_hits.iter_rows(named=True):
        qacc = parse_accession(row["query_name"])
        tacc = parse_accession(row["match_name"])
        if qacc == tacc:
            continue
        gene_species.setdefault(qacc, (row["query_taxid"], row["query_species"]))
        gene_species.setdefault(tacc, (row["target_taxid"], row["target_species"]))
        uf.union(qacc, tacc)

    # Collect connected components
    components = defaultdict(list)
    for acc in gene_species:
        components[uf.find(acc)].append(acc)

    # Only keep multi-species groups
    ortholog_groups = [
        sorted(accs)
        for accs in components.values()
        if len({gene_species[a][0] for a in accs}) >= 2
    ]

    print(f"Proteins in groups:      {sum(len(g) for g in components.values()):,}")
    print(f"Ortholog groups (≥2 sp): {len(ortholog_groups):,}")

    # Group size distribution
    group_sizes = [len(g) for g in ortholog_groups]
    species_per_group = [len({gene_species[a][0] for a in g}) for g in ortholog_groups]

    print(f"Median group size:       {np.median(group_sizes):.0f}")
    print(f"Median species per group:{np.median(species_per_group):.0f}")
    print(
        f"Max species per group:   {max(species_per_group) if species_per_group else 0}"
    )

In [ ]:
if len(strong_hits) > 0 and ortholog_groups:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(
        f"Ortholog Groups (max_containment >= {CONTAINMENT_THRESHOLD}, ≥2 species)",
        fontsize=11,
    )

    ax = axes[0]
    ax.hist(
        group_sizes,
        bins=min(50, max(group_sizes)),
        color="steelblue",
        edgecolor="white",
        log=True,
    )
    ax.set_xlabel("Group size (# proteins)")
    ax.set_ylabel("Count (log scale)")
    ax.set_title(f"Group size distribution\n(n={len(ortholog_groups):,} groups)")

    ax = axes[1]
    ax.hist(
        species_per_group,
        bins=range(2, max(species_per_group) + 2),
        color="darkorange",
        edgecolor="white",
        log=True,
    )
    ax.set_xlabel("Species per group")
    ax.set_ylabel("Count (log scale)")
    ax.set_title("Species per ortholog group")

    plt.tight_layout()
    plt.savefig(
        OUT_DIR / f"{PREFIX}ortholog_group_sizes.png", dpi=150, bbox_inches="tight"
    )
    plt.show()

## 9. Most Universal Groups (Widest Phylogenetic Spread)

Show the groups that span the most species — candidates for deeply conserved proteins.

In [ ]:
if len(strong_hits) > 0 and ortholog_groups:
    group_records = [
        {
            "group_id": i,
            "n_proteins": len(g),
            "n_species": len({gene_species[a][0] for a in g}),
            "species_list": "; ".join(sorted({gene_species[a][1] for a in g})),
            "representative_proteins": "; ".join(g[:5]),
        }
        for i, g in enumerate(ortholog_groups)
    ]

    groups_df = pl.DataFrame(group_records)

    print("Top 20 most universal groups:")
    display(
        groups_df.sort("n_species", descending=True)
        .head(20)
        .select(
            [
                "group_id",
                "n_species",
                "n_proteins",
                "species_list",
                "representative_proteins",
            ]
        )
    )

    groups_csv_path = OUT_DIR / f"{PREFIX}ortholog_groups.csv"
    groups_df.write_csv(str(groups_csv_path))
    print(f"\nSaved {len(groups_df):,} groups to {groups_csv_path}")

## 10. Summary and Findings

In [ ]:
print("=" * 60)
print("QFO 2020 KMERSEEK ANALYSIS SUMMARY")
print(f"Parameters: k={KSIZE}, moltype={MOLTYPE}, scaled=1")
print("=" * 60)
print()
print("WHY ORTHOXML IS EMPTY:")
print("  format_orthoxml.py looks for columns:")
print("    'poisson_pvalue', 'prob_overlap', 'prob_overlap_adjusted'")
print("  kmerseek search CSVs have: containment, jaccard, ANI, intersect_hashes")
print("  → No p-value column found → all files skipped → empty <groups />")
print()
print("FIX NEEDED:")
print("  format_orthoxml.py must be updated to accept kmerseek containment scores")
print(
    "  or a poisson_pvalue must be computed from (intersect_hashes, expected_shared_kmers)"
)
print()
print("DATA SUMMARY (using containment >= 0.10 as proxy significance):")
if len(strong_hits) > 0:
    print(f"  Strong hits (max_containment >= 0.10): {len(strong_hits):,}")
    print(f"  Ortholog groups formed:                {len(ortholog_groups):,}")
    print(f"  Median group size:                     {np.median(group_sizes):.0f}")
    print(f"  Most species-wide group spans:         {max(species_per_group)} species")
else:
    print("  (Run cells above to populate)")